In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import preprocessing
from sklearn.model_selection import train_test_split

In [59]:
import importlib

importlib.reload(preprocessing)

<module 'preprocessing' from '/home/julien/Projets/churn-prediction/notebooks/../src/preprocessing.py'>

In [ ]:
from pathlib import Path

BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR / "src"))

In [61]:
from preprocessing import (
    add_charge_level,
    add_engagement_score,
    add_service_count,
    add_tenure_group
)

In [62]:
dff = pd.read_csv("../data/Telco_customer_churn.csv")
dff.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [63]:
dff.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [64]:
df = dff.drop(columns=["customerID", "gender", "PhoneService"])


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [66]:
df = df.dropna()

In [67]:
df.head(1)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Yes,No,1,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No


In [68]:
df = add_tenure_group(df)
df = add_charge_level(df)
df = add_engagement_score(df)
df = add_service_count(df)

J'ai crée plusieurs nouvelles features que je pense pertinente, je les ai écrites dans les dossier preprocessing pour pouvoir m'en reservir au besoin.

In [69]:
df.head(1)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,high_charges,engagement_score,service_count
0,0,Yes,No,1,No phone service,DSL,No,Yes,No,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-1y,0,29.85,1


In [70]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [71]:
y = y.map({"Yes": 1, "No": 0})

In [72]:
X = pd.get_dummies(X, drop_first=True)

In [73]:
X = X.astype("float64")

In [74]:
X.head(5)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,high_charges,engagement_score,service_count,Partner_Yes,Dependents_Yes,MultipleLines_No phone service,...,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_1-2y,tenure_group_2-4y,tenure_group_4-6y
0,0.0,1.0,29.85,29.85,0.0,29.85,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,34.0,56.95,1889.50,0.0,1936.30,2.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,0.0,2.0,53.85,108.15,0.0,107.70,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.0,45.0,42.30,1840.75,0.0,1903.50,3.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,2.0,70.70,151.65,1.0,141.40,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
